# Cryogenic Roofline Calculator, version 2.1.0

Companion to *The Cryogenic Roofline: Energy Break-Even for Quantum-Assisted LLM Serving*.

Author: Somdip Dey. ORCID: https://orcid.org/0000-0001-6161-4637.

This notebook implements a conditional architectural requirements analysis. It does not rank qubit modalities, report an LLM benchmark, or establish quantum advantage. The measured anchor is 23 mW per qubit under active 4 K control (Underwood et al., PRX Quantum 5, 010326, 2024; https://doi.org/10.1103/PRXQuantum.5.010326). Linear eight-qubit scaling, activity, timing, estimator precision, scheduling, efficiency and plant allocation are separate assumptions.

Keep this notebook beside `artifact/`. Run all cells from the extracted package directory. Python 3.12, NumPy and Matplotlib suffice. The exact tested versions are in `artifact/requirements.txt`. The same shared model generates all figures, CSVs and LaTeX numeric inputs.


In [1]:
from pathlib import Path
import sys, json, platform
ROOT = Path.cwd()
if not (ROOT / 'artifact' / 'cryogenic_model.py').exists():
    raise RuntimeError('Start this notebook in the extracted package directory containing artifact/.')
sys.path.insert(0, str(ROOT / 'artifact'))
import cryogenic_model as model
import validate
config = json.loads((ROOT / 'artifact' / 'config.json').read_text())
print('Python', platform.python_version())
print(json.dumps(config, indent=2))


Python 3.12.14
{
  "artifact_version": "2.1.0",
  "hot_temperature_k": 300.0,
  "controller_temperature_k": 4.0,
  "carnot_fraction": 0.02,
  "qubits": 8,
  "controller_power_per_qubit_w": 0.023,
  "shot_duration_s": 0.0001,
  "failure_probability": 0.05,
  "statistical_error": 0.1,
  "allowed_bias": 0.02,
  "tokens_per_decision": 32,
  "availability": 0.9,
  "token_rate_per_s": 1000.0,
  "available_controller_cooling_w_per_plant": 1.5,
  "fixed_wall_power_w_per_plant": 5000.0,
  "illustrative_energy_fraction": 0.3,
  "illustrative_overhead_fraction": 0.05
}


## 1. A consistent energy ledger

Cooling work per lifted joule is $\chi=(T_h-T_c)/(\eta T_c)$. For local electrical dissipation, total electrical input plus cooling has thermal amplification factor (TAF) $A=1+\chi$. At ambient $\chi=0$ and $A=1$.

The general ledger adds all electrical input once, incremental heat-lifting work for each stage, and separately allocated fixed wall power. Stage cooling factors must not each contain the full common plant power. Absorbed warm-generated RF or optical power creates cold heat but is not another independent electrical supply.


In [2]:
for temperature, eta in [(300, .02), (77, .15), (4, .02), (1, .02), (.02, .02)]:
    print(temperature, 'K: chi =', model.cooling_factor(temperature, eta),
          '; A =', model.electrical_multiplier(temperature, eta))
# 0.1 J supplied warm; 0.001 J of it dissipates at 4 K.
print('Example total input + cooling:', model.added_wall_energy(.1, [(4, .02, .001)]), 'J')


300 K: chi = 0.0 ; A = 1.0
77 K: chi = 19.30735930735931 ; A = 20.30735930735931
4 K: chi = 3700.0 ; A = 3701.0
1 K: chi = 14950.0 ; A = 14951.0
0.02 K: chi = 749950.0 ; A = 749951.0
Example total input + cooling: 3.8000000000000003 J


## 2. Useful output and precision

The synthetic eight-qubit reference circuit uses 8 input $R_y$ gates, then four layers of $R_y$, $R_z$, and a sequential CNOT ring. It returns $z=\langle Z_0\rangle$. The implementation specifies all angles and ordering. It is a small, classically tractable score, not a trained serving policy.

For independent outcomes in $[-1,1]$, the sufficient Hoeffding budget is $S=\lceil 2\ln(2/\delta)/\epsilon^2\rceil$. This bounds statistical error around the implemented mean. A separately certified hardware bias bound $\beta$ is required for error at most $\epsilon+\beta$ against the ideal reference. No hardware bias certification or classical energy measurement is supplied.


In [3]:
reference = model.reference_score()
print(json.dumps(reference, indent=2))
# Variable inputs; the default vector is a verification fixture, not useful repeated work.
import numpy as np
for offset in [-.2, 0, .2]:
    print(offset, model.reference_score(np.linspace(-.6, .8, 8) + offset)["ideal_z0"])
for epsilon in [.2, .1, .05]:
    print('Statistical half-width', epsilon, ':', model.shot_count(epsilon), 'shots')


{
  "input_angles_rad": [
    -0.6,
    -0.4,
    -0.2,
    0.0,
    0.19999999999999996,
    0.3999999999999999,
    0.6,
    0.8
  ],
  "qubits": 8,
  "amplitudes": 256,
  "single_qubit_gates": 72,
  "cnot_gates": 32,
  "norm": 0.9999999999999999,
  "ideal_z0": 0.5071208833939499,
  "hardware_bias_certified": false,
  "classical_energy_measured": false
}
-0.2 0.5084550246136204
0 0.5071208833939499
0.2 0.44384725081001314
Statistical half-width 0.2 : 185 shots
Statistical half-width 0.1 : 738 shots
Statistical half-width 0.05 : 2952 shots


## 3. Controller contribution and service capacity

The case charges 23 mW per active qubit for an assumed full 100-microsecond shot, including preparation, execution, reset and readout time. Power consumed by other preparation/readout/host components remains unquantified. The eight-qubit allocation is an extrapolation, not a measured eight-qubit controller or a validated pulse schedule.

A 90% availability factor reduces service capacity; it does not erase energy for completed work. Engines are replicated to handle completed tokens/s at one decision per 32 tokens. A 1.5 W controller cooling allocation per plant and 5 kW fixed wall power per plant are illustrative inputs, not product specifications. Controller idle power is set to zero in this partial calculation. The complete system requires additional measurements.


In [4]:
for epsilon in [.2, .1, .05]:
    print(json.dumps(model.case_result(config, epsilon=epsilon), indent=2))


{
  "epsilon_stat": 0.2,
  "delta": 0.05,
  "shots": 185,
  "decision_duration_s": 0.018500000000000003,
  "controller_power_w": 0.184,
  "controller_multiplier": 3701.0,
  "controller_energy_j_per_decision": 12.598204000000003,
  "controller_energy_j_per_token": 0.3936938750000001,
  "engine_capacity_decisions_per_s": 48.648648648648646,
  "rate_tokens_per_s": 1000.0,
  "engines_required": 1,
  "engines_per_plant": 8,
  "plants_required": 1,
  "controller_peak_w_all_engines": 0.184,
  "allocated_fixed_j_per_token": 5.0,
  "controller_plus_fixed_j_per_token": 5.393693875,
  "scope": "controller contribution and optional assumed fixed plant allocation; other loads unquantified"
}
{
  "epsilon_stat": 0.1,
  "delta": 0.05,
  "shots": 738,
  "decision_duration_s": 0.0738,
  "controller_power_w": 0.184,
  "controller_multiplier": 3701.0,
  "controller_energy_j_per_decision": 50.2566192,
  "controller_energy_j_per_token": 1.57051935,
  "engine_capacity_decisions_per_s": 12.195121951219512,
 

In [5]:
for rate in [10, 100, 1000, 3000, 3200, 10000]:
    r = model.case_result(config, rate=rate)
    print(rate, 'tokens/s:', r['engines_required'], 'engines,', r['plants_required'],
          'plants, controller + fixed subtotal', r['controller_plus_fixed_j_per_token'], 'J/token')


10 tokens/s: 1 engines, 1 plants, controller + fixed subtotal 501.57051935 J/token
100 tokens/s: 1 engines, 1 plants, controller + fixed subtotal 51.57051935 J/token
1000 tokens/s: 3 engines, 1 plants, controller + fixed subtotal 6.57051935 J/token
3000 tokens/s: 8 engines, 1 plants, controller + fixed subtotal 3.2371860166666666 J/token
3200 tokens/s: 9 engines, 2 plants, controller + fixed subtotal 4.69551935 J/token
10000 tokens/s: 26 engines, 4 plants, controller + fixed subtotal 3.57051935 J/token


## 4. Custom scenarios

Edit `artifact/config.json` to explore alternate scenarios and regenerate outputs. Figure labels and CSVs follow the configuration. Static manuscript prose and captions still require corresponding author edits. The reference circuit always has eight qubits, independently of controller sizing overrides. This exploratory copy below changes a parameter without overwriting the manuscript configuration. Changing only a shot budget is not permitted here: it is recalculated from the precision requirement. A useful application would also measure the best classical implementation of the same output contract.


In [6]:
custom = dict(config)
custom['shot_duration_s'] = 50e-6
print(json.dumps(model.case_result(custom), indent=2))


{
  "epsilon_stat": 0.1,
  "delta": 0.05,
  "shots": 738,
  "decision_duration_s": 0.0369,
  "controller_power_w": 0.184,
  "controller_multiplier": 3701.0,
  "controller_energy_j_per_decision": 25.1283096,
  "controller_energy_j_per_token": 0.785259675,
  "engine_capacity_decisions_per_s": 24.390243902439025,
  "rate_tokens_per_s": 1000.0,
  "engines_required": 2,
  "engines_per_plant": 8,
  "plants_required": 1,
  "controller_peak_w_all_engines": 0.368,
  "allocated_fixed_j_per_token": 5.0,
  "controller_plus_fixed_j_per_token": 5.785259675,
  "scope": "controller contribution and optional assumed fixed plant allocation; other loads unquantified"
}


## 5. Regenerate the exact manuscript artifacts

The build uses the configuration file, not the exploratory copy above. It writes all three figures as vector PDFs and PNGs, numerical macros, table rows, CSVs, and the reference score. `table_controller_requirements.csv` records the precision sweep; `table_capacity.csv` records the capacity examples. Obsolete ranking and break-even-map filenames are no longer emitted.


In [7]:
result = model.build(ROOT)
print(json.dumps(result, indent=2))


{
  "epsilon_stat": 0.1,
  "delta": 0.05,
  "shots": 738,
  "decision_duration_s": 0.0738,
  "controller_power_w": 0.184,
  "controller_multiplier": 3701.0,
  "controller_energy_j_per_decision": 50.2566192,
  "controller_energy_j_per_token": 1.57051935,
  "engine_capacity_decisions_per_s": 12.195121951219512,
  "rate_tokens_per_s": 1000.0,
  "engines_required": 3,
  "engines_per_plant": 8,
  "plants_required": 1,
  "controller_peak_w_all_engines": 0.552,
  "allocated_fixed_j_per_token": 5.0,
  "controller_plus_fixed_j_per_token": 6.57051935,
  "scope": "controller contribution and optional assumed fixed plant allocation; other loads unquantified"
}


## 6. Validation

Checks cover stage accounting, ambient behavior, the Hoeffding guarantee, break-even with overhead, engine and cooling-allocation constraints, reference-state normalization and invalid inputs. No test assumes that a modality must win, lose or be physically impossible.


In [8]:
validate.run()


PASS: energy accounting, shot guarantee, crossover, capacity, reference circuit and input domains


## Interpretation and limits

The output is a conditional controller energy requirement. Complete energy must add missing host, control idle, wiring, detection, other-stage heat, calibration and error-management costs without double counting. Meeting the modeled 4 K allocation does not prove feasibility at other stages. Capacity is not a tail-latency guarantee. The estimator confidence statement assumes independent samples and does not certify hardware bias.

No public repository, DOI, measured LLM energy saving, or trained routing-policy benefit is asserted. Submit this notebook with the shared Python model and configuration as supplementary material.
